In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
df = pd.read_csv(os.path.join(os.getcwd(),'Data/master_data.csv'))

In [3]:
df.columns

Index(['Ticker', 'TradeDate', 'ExpiryDate', 'CallPut', 'AdjStrike', 'Strike',
       'AdjSpot', 'Spot', 'Month_To_Expiry', 'days_to_expiry', 'ExpirySpot',
       'AdjExpiry', 'Moneyness', 'AbsMoneyness', 'Symbol', 'LastTradePrice',
       'BidPrice', 'AskPrice', 'BidImpliedVolatility', 'AskImpliedVolatility',
       'OpenInterest', 'Volume', 'Delta', 'Gamma', 'Vega', 'Theta', 'Rho',
       'FmtTradeDate', 'FmtExpiryDate', '4 weeks coupon equivalent',
       '8 weeks coupon equivalent', '13 weeks coupon equivalent',
       '17 weeks coupon equivalent', '26 weeks coupon equivalent',
       '52 weeks coupon equivalent', 'risk_free_rate_4', 'risk_free_rate_8',
       'risk_free_rate_13', 'risk_free_rate_17', 'risk_free_rate_26',
       'risk_free_rate_52'],
      dtype='object')

In [4]:
def sort_data(df,no_of_exp_dates = None):
    Date = []
    Expiry = []
    CallPut = []
    Str_Cnt = []
    
    for date in df.TradeDate.unique():
        for c_p in ['c','p']:
            df_by_trade = df[(df.TradeDate==date)&(df.CallPut==c_p)][['TradeDate','ExpiryDate','CallPut','AdjStrike']]
            exp_dates = np.sort(df_by_trade.ExpiryDate.unique())
            exp_frame = pd.DataFrame(df_by_trade.ExpiryDate.value_counts()).reset_index()
            exp_frame =exp_frame.rename(columns={'count':'strike_count'})

            if no_of_exp_dates:
                iter_count = no_of_exp_dates
            else:
                iter_count = len(exp_dates)
            for i in range(iter_count):
                strikes = exp_frame[exp_frame.ExpiryDate==exp_dates[i]].strike_count.values[0]    
                Date.append(date)  
                Expiry.append(exp_dates[i])
                CallPut.append(c_p)
                Str_Cnt.append(strikes)
        
            
    df_sorted = pd.DataFrame({'TradeDate':Date,'ExpiryDate':Expiry,'OptionType':CallPut,'StrikeCount':Str_Cnt})
    df_sorted = df_sorted.sort_values(['TradeDate','ExpiryDate','OptionType'])
    df_sorted['FmtExpiryDate'] = pd.to_datetime(df_sorted["ExpiryDate"], format="%Y%m%d")
    df_sorted['FmtTradeDate'] = pd.to_datetime(df_sorted["TradeDate"], format="%Y%m%d")
    df_sorted["ExpDays"] = (df_sorted.FmtExpiryDate-df_sorted.FmtTradeDate).dt.days
    df_sorted['ExpMonths'] = (df_sorted.FmtExpiryDate.dt.year - df_sorted.FmtTradeDate.dt.year) * 12 + (df_sorted.FmtExpiryDate.dt.month - df_sorted.FmtTradeDate.dt.month)
    df_sorted=df_sorted.drop(columns=['FmtExpiryDate','FmtTradeDate'])
    df_sorted = df_sorted[['TradeDate','ExpiryDate','ExpMonths','ExpDays','OptionType','StrikeCount']]
   
    
    return df_sorted
    

    

In [5]:
df_sorted = sort_data(df,no_of_exp_dates=3)


In [8]:
df_sorted[df_sorted.TradeDate==20100226]

,TradeDate,ExpiryDate,ExpMonths,ExpDays,OptionType,StrikeCount
18,20100226,20100320,1,22,c,26
21,20100226,20100320,1,22,p,26
19,20100226,20100417,2,50,c,35
22,20100226,20100417,2,50,p,35
20,20100226,20100717,5,141,c,34
23,20100226,20100717,5,141,p,34


In [9]:
len(df_sorted)

2304

In [10]:
def data_extractor(trade_date,expiry,option_type):
    df_stk = df[(df.TradeDate==trade_date)&(df.ExpiryDate==expiry)&(df.CallPut==option_type)]
    df_stk = df_stk[['TradeDate', 'ExpiryDate', 'CallPut','AdjStrike', 'Strike', 'AdjSpot','Spot',
                     'Month_To_Expiry', 'days_to_expiry', 'ExpirySpot','AdjExpiry', 'Moneyness',
                     'AbsMoneyness', 'Symbol',]]
    return df_stk

In [12]:
df_stk = data_extractor(20100430,20100522,'c').sort_values(by = 'AdjStrike')
df_stk

,TradeDate,ExpiryDate,CallPut,AdjStrike,Strike,AdjSpot,Spot,Month_To_Expiry,days_to_expiry,ExpirySpot,AdjExpiry,Moneyness,AbsMoneyness,Symbol
2908,20100430,20100522,c,5.00,140.0,9.32,261.09,1,22,8.65,2010-05-21,50.0,53.648069,EV
2907,20100430,20100522,c,5.18,145.0,9.32,261.09,1,22,8.65,2010-05-21,60.0,55.579399,EV
2906,20100430,20100522,c,5.36,150.0,9.32,261.09,1,22,8.65,2010-05-21,60.0,57.510730,EV
2937,20100430,20100522,c,5.54,155.0,9.32,261.09,1,22,8.65,2010-05-21,60.0,59.442060,EV
3381,20100430,20100522,c,5.71,160.0,9.32,261.09,1,22,8.65,2010-05-21,60.0,61.266094,EV
3380,20100430,20100522,c,5.89,165.0,9.32,261.09,1,22,8.65,2010-05-21,60.0,63.197425,EV
3372,20100430,20100522,c,6.07,170.0,9.32,261.09,1,22,8.65,2010-05-21,70.0,65.128755,EV
3371,20100430,20100522,c,6.25,175.0,9.32,261.09,1,22,8.65,2010-05-21,70.0,67.060086,EV
2921,20100430,20100522,c,6.43,180.0,9.32,261.09,1,22,8.65,2010-05-21,70.0,68.991416,EV
3365,20100430,20100522,c,6.61,185.0,9.32,261.09,1,22,8.65,2010-05-21,70.0,70.922747,EV


In [13]:
df.Symbol.value_counts()

Symbol
SP    343016
EV    340307
Name: count, dtype: int64

In [14]:
np.sort(df[df.TradeDate==20251231].ExpiryDate.unique())

array([20260102, 20260109, 20260116, 20260123, 20260130, 20260206,
       20260213, 20260220, 20260320, 20260417, 20260515, 20260618,
       20260717, 20260821, 20260918, 20261218, 20270115, 20270617,
       20271217, 20280121, 20280317])